In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [ ]:
df_new = pd.read_csv('OctNov_CarData.csv')
df_old = pd.read_csv('total_call_data.csv')
df_new.shape

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\2363734591.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df_old = pd.read_csv('total_call_data.csv')


,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
0,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
1,9f5be247-1027-4443-bfe1-86a740f85b8d,NaN,FarmworkerMain,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
2,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-10-01 00:25:00,NaN,NaN,NaN,0
3,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,FarmworkerMain,NaN,2025-10-01 00:25:00,NaN,NaN,NaN,0
4,9f5be247-1027-4443-bfe1-86a740f85b8d,Farmworker Main Number Telephony EP,NaN,FarmworkerMainMenu,2025-10-01 00:25:00,NaN,NaN,NaN,0


In [55]:
# Split old data set into before and after Oct 15
df_old['Activity Start Timestamp'] = pd.to_datetime(df_old['Activity Start Timestamp'], format='mixed')
df_new['Activity Start Timestamp'] = pd.to_datetime(df_new['Activity Start Timestamp'], format='mixed')

df_new_before = df_new[df_new['Activity Start Timestamp'] <='2025-10-15']
df_new = df_new[df_new['Activity Start Timestamp'] > '2025-10-15 23:59:59']
df_old = pd.concat([df_old, df_new_before])

C:\Users\Owner\AppData\Local\Temp\ipykernel_37484\83288708.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['Activity Start Timestamp'] = pd.to_datetime(df_new['Activity Start Timestamp'], format='mixed')


In [56]:
df_new['Queue Name'].value_counts()

Queue Name
Staff Directory English Transfer        8610
Clinic Voicemail Transfer               7608
Front Desk Transfer                     4926
Intake Outdial Queue                    2524
Family                                  1422
Consumer                                1241
Housing                                  852
Benefits                                 812
Staff Directory Spanish Transfer         807
Criminal Records Voicemail Transfer      673
SubSenior Family                         397
SubSenior Consumer                       378
SubSenior Benefits                       313
SubSenior Tenant                         281
Employment                               280
SubSenior Homeowner                      257
SubSenior ADAPT                          257
HIV Voicemail Transfer                   135
Family SP                                118
ADAPT                                    102
Immigration SP                            81
SubSenior Employment                      76

In [57]:
df_new.shape

(243493, 9)

In [58]:
#Filter dataframe
# Steps, check all rows for an ID
# If all the Queue Names for that ID are either na or contain 'Transfer', drop that ID entirely
def filter_transfer_only(df):
    ids_to_drop = []
    for contact_id, group in df.groupby('Contact Session ID'):
        queue_names = group['Queue Name'].dropna().astype(str)
        if all('transfer' in name.lower() or 'intake outdial' in name.lower() for name in queue_names) or queue_names.empty:
            ids_to_drop.append(contact_id)
    filtered_df = df[~df['Contact Session ID'].isin(ids_to_drop)]
    return filtered_df

df_new_filt = filter_transfer_only(df_new)
df_old_filt = filter_transfer_only(df_old)

In [59]:
df_new.shape, df_new_filt.shape, df_old.shape, df_old_filt.shape

((243493, 9), (30889, 9), (3301965, 9), (420102, 9))

In [60]:
df_new_filt = df_new.copy()
df_old_filt =df_old.copy()

In [76]:
# First Step: Check if proportion of senior menu queues is the same in both datasets
# Filter out all queues that contain transfer or intake outdial

ids_senior_new = df_new[df_new['Activity Name'].str.contains('Senior', na=False)]['Contact Session ID'].unique()
ids_senior_old = df_old[df_old['Activity Name'].str.contains('Senior', na=False)]['Contact Session ID'].unique()

# Also check for legalmenu
ids_senior_new2 = df_new[df_new['Activity Name'].str.contains('LegalMenu', na=False)]['Contact Session ID'].unique()
ids_senior_old2 = df_old[df_old['Activity Name'].str.contains('LegalMenu', na=False)]['Contact Session ID'].unique()

ids_new = set(list(ids_senior_new) + list(ids_senior_new2))
ids_old = set(list(ids_senior_old) + list(ids_senior_old2))

prop_senior_new = len(ids_new) / df_new['Contact Session ID'].nunique()
prop_senior_old = len(ids_old) / df_old['Contact Session ID'].nunique()

print(f"prop new: {prop_senior_new * 100:.4f} %, prop old: {prop_senior_old* 100:.4f} %")

prop new: 45.4317 %, prop old: 43.8742 %


After some explatory analysis, (which can and should be double checked) it appears that in Activity Name, before in all instances before the new implementation SeniorsMenu shows up prior to LegalMenu1, while in instances after the implementation SeniorsMenu now shows up after LegalMenu1(and 2). 

When looking for the occurence of both a LegalMenu1/2 and SeniorsMenu, the old dataset has about 43.8% of people reaching SeniorsMenu while the new dataset has a little more at about 45.4%. This is in line with observations made by the employees siting more people reaching senior menus by accident. 

Next Steps: The goal now is to find the amonut of time it takes for a customer who gets to Senior menu to make a call. This can be found in a handful of ways. 

Method 1: Finding total length of calls for all these people. This is easiest method, but leads to the length of time waiitng in queue or being on call with an agent to have a potentially large impact.

Method 2: Find length of call from start until we get to PreQueueMessage or ClosedQueue. This takes away any waiting or speaking bias, but it potentially will leave out some people if they never end up in a queue. I am not sure how many people(if any) will be left out by this determination. A possible solution is simply to take time of full call for those who never get to a queue or closedqueue. 

Method 3: Taking time from start of call to LegalMenu1. I am not sure, but even though SeniorsMenu shows up after LegalMenu1 in the new dataset there is a chance that that is a placeholder and does not actually take up anytime. Will have to check how times are recorded for both before and after implemenation and see if this idea seems ot make any sense or not. 

In [62]:
temp = df_new_filt[df_new_filt['Contact Session ID'].isin(ids_senior_new)]
temp['Queue Name'].value_counts()

Queue Name
SubSenior Family         397
SubSenior Consumer       378
SubSenior Benefits       313
SubSenior Tenant         281
SubSenior ADAPT          257
SubSenior Homeowner      257
SubSenior Employment      76
SubSenior Family SP       52
SubSenior Consumer SP     49
SubSenior Benefits SP     23
SubSenior ADAPT SP         3
Name: count, dtype: int64

In [68]:
df_old.loc[df_old['Activity Name'] == 'ClosedQueueMenu']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
14,001a3748-8d50-4550-8461-33547983deb0,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-01-14 12:43:16,NaN,NaN,NaN,12
15,001a3748-8d50-4550-8461-33547983deb0,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-01-14 12:44:33,NaN,NaN,NaN,12
31,00229391-8614-4eb1-b1e0-e0a78076e0e4,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-01-14 14:52:42,NaN,NaN,NaN,14
33,00229391-8614-4eb1-b1e0-e0a78076e0e4,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-01-14 14:53:10,NaN,NaN,NaN,14
69,0042f5e6-6c86-4bc1-84a5-4bfee8eb5580,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-01-14 13:20:27,NaN,NaN,NaN,13
...,...,...,...,...,...,...,...,...,...
187295,528d69ab-236d-4416-b803-0a7caa58c7cf,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-10-14 16:04:00,NaN,NaN,NaN,16
187298,81d2e4f8-7b94-41e8-8fbd-4aa98aef8047,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-10-14 16:04:00,NaN,NaN,NaN,16
187320,7276d3b4-62c6-4b75-85d5-2b09eeca3b39,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-10-14 16:05:00,NaN,NaN,NaN,16
187386,848b2ade-31bf-4e7b-a634-c5350974341b,Closed Queue Menu Telephony EP,NaN,ClosedQueueMenu,2025-10-14 16:08:00,NaN,NaN,NaN,16


In [66]:
df_new.loc[df_new['Contact Session ID'] == '2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
50447,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50448,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,LACMain,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50449,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-10-16 08:00:06,NaN,NaN,NaN,8
50450,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,LACMain,NaN,2025-10-16 08:00:06,NaN,NaN,NaN,8
50467,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Main Number Telephony EP,NaN,MainMenu,2025-10-16 08:00:17,NaN,NaN,NaN,8
50471,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,LegalMenu,NaN,2025-10-16 08:00:24,NaN,NaN,NaN,8
50472,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Legal Menu Telephony EP,NaN,LegalMenu1,2025-10-16 08:00:24,NaN,NaN,NaN,8
50510,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Legal Menu Telephony EP,NaN,LegalMenu2,2025-10-16 08:01:00,NaN,NaN,NaN,8
50544,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,NaN,SeniorsMenu,NaN,2025-10-16 08:02:07,NaN,NaN,NaN,8
50545,2f9a4103-77a7-4ddd-9647-2b2f89e2bbc7,Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-10-16 08:02:07,NaN,NaN,NaN,8


In [70]:
df_old.loc[df_old['Contact Session ID'] == '001a3748-8d50-4550-8461-33547983deb0']

,Contact Session ID,EP Name,Flow Name,Activity Name,Activity Start Timestamp,Queue Name,Agent Name,Termination Reason,hour
0,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
1,001a3748-8d50-4550-8461-33547983deb0,NaN,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
2,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,LanguageSelectionMenu,2025-01-14 12:39:32,NaN,NaN,NaN,12
3,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,LACMain,NaN,2025-01-14 12:39:32,NaN,NaN,NaN,12
4,001a3748-8d50-4550-8461-33547983deb0,Main Number Telephony EP,NaN,MainMenu,2025-01-14 12:39:45,NaN,NaN,NaN,12
5,001a3748-8d50-4550-8461-33547983deb0,NaN,PreLegalMenuSeniorsMenu,NaN,2025-01-14 12:40:09,NaN,NaN,NaN,12
6,001a3748-8d50-4550-8461-33547983deb0,Pre-Legal Menu Seniors Menu Telephony EP,NaN,SeniorsMenu,2025-01-14 12:40:09,NaN,NaN,NaN,12
7,001a3748-8d50-4550-8461-33547983deb0,NaN,LegalMenu,NaN,2025-01-14 12:40:18,NaN,NaN,NaN,12
8,001a3748-8d50-4550-8461-33547983deb0,Legal Menu Telephony EP,NaN,LegalMenu1,2025-01-14 12:40:18,NaN,NaN,NaN,12
9,001a3748-8d50-4550-8461-33547983deb0,Legal Menu Telephony EP,NaN,LegalMenu2,2025-01-14 12:40:54,NaN,NaN,NaN,12
